In [61]:
# notebook debugging preamble
# %load_ext autoreload
# %autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets


# 定义波形生成函数
def wave(freq, phase, t):
    return np.cos(2 * np.pi * freq * t + phase)


# 初始化参数
duration = 50
period = 10
freq = 1 / period
sample_rate = 1000
t = np.linspace(0, duration, sample_rate * duration)
y = wave(freq, 0, t)

# 创建输出区域用于显示控制信息和图表
output = widgets.Output()
spectrum_output = widgets.Output()
phase_output = widgets.Output()

# 初始化采样参数
sampling_rate = 1
sampling_duration_equal_to_period = 10  # 初始值

# 创建摇杆控件
# 创建滑块用于微调
slider = widgets.IntSlider(
    value=sampling_duration_equal_to_period,
    min=5,
    max=50,
    step=1,
    description="采样点数",
)
slider.layout.width = "500px"


def update_slider(
    sampling_duration,
):
    freq_bins, magnitude_spectrum, phase_spectrum = compute_spectrum(
        sampling_rate, sampling_duration, freq
    )

    # 清空并更新输出
    with output:
        output.clear_output(wait=True)
        print(f"当前采样点数: {sampling_duration}")

    with spectrum_output:
        spectrum_output.clear_output(wait=True)
        plot_magnitude_spectrum(freq_bins, magnitude_spectrum)

    with phase_output:
        phase_output.clear_output(wait=True)
        plot_phase_spectrum(freq_bins, phase_spectrum, deg=True)


def plot_wrapper(
    sampling_duration,
):
    freq_bins, magnitude_spectrum, phase_spectrum, spectrum = compute_spectrum(
        sampling_rate, sampling_duration, freq
    )
    fig, ax = plt.subplots(2, 2, figsize=(15, 10))

    plot_magnitude_spectrum(ax[0, 0], freq_bins, magnitude_spectrum)
    plot_phase_spectrum(ax[0, 1], freq_bins, phase_spectrum, deg=True)
    # plot_duration = 50
    # t = np.linspace(0, plot_duration - 1, sampling_rate * plot_duration)
    plot_reconstruct_signal(ax[1, 0], t, y, spectrum)
    ax[1, 1] = None


def interactive_update(sampling_duration_equal_to_period):
    plot_wrapper(
        sampling_duration=sampling_duration_equal_to_period,
    )


# 计算采样点和频谱
def compute_spectrum(sampling_rate, sampling_duration, freq):
    samples = sampling_rate * sampling_duration
    t_sample = np.linspace(0, sampling_duration - 1, samples)
    y_sample = wave(freq, 0, t_sample)
    spectrum = normalizing(np.fft.fft(y_sample))
    one_side_spectrum = spectrum[: samples // 2]
    magnitude_spectrum = np.abs(one_side_spectrum)
    # magnitude_spectrum_normalized = maximum_normalization(magnitude_spectrum)
    magnitude_spectrum = magnitude_spectrum / len(magnitude_spectrum)
    phase_spectrum = np.angle(one_side_spectrum, deg=True)
    freq_bins = np.fft.fftfreq(len(t_sample), d=1 / sampling_rate)[: samples // 2]
    return freq_bins, magnitude_spectrum, phase_spectrum, spectrum


# 辅助函数
def normalizing(x: np.array):
    norm = np.abs(x).sum()
    th = norm * 0.01
    small_idx_real = np.where(np.abs(x.real) < th)
    x.real[small_idx_real] = 0
    small_idx_imag = np.where(np.abs(x.imag) < th)
    x.imag[small_idx_imag] = 0
    return x


def maximum_normalization(x):
    return x / np.max(x)


def plot_magnitude_spectrum(ax, freq_bins, spectrum):
    ax.set_title("Magnitude Spectrum")
    ax.set_xlabel("Frequency (Hz)")
    # ax.set_xlim(0, max(freq_bins))
    # ax.set_xticks(freq_bins)
    ax.set_xlim(0, sampling_rate / 2)
    ax.set_xticks(np.linspace(0, sampling_rate / 2, 11))
    ax.set_ylabel("Magnitude")
    ax.set_ylim(0, 1.2)
    markerline, stemline, baseline = ax.stem(
        freq_bins,
        spectrum,
        linefmt="r-",
        markerfmt="o",
        basefmt=" ",
    )
    markerline.set_markerfacecolor("none")


def plot_phase_spectrum(ax, freq_bins, phase_spectrum, deg=False):
    ax.set_title("Phase Spectrum")
    ax.set_xlabel("Frequency (Hz)")
    # ax.set_xlim(0, max(freq_bins))
    # ax.set_xticks(freq_bins)
    ax.set_xlim(0, sampling_rate / 2)
    ax.set_xticks(np.linspace(0, sampling_rate / 2, 11))
    if deg:
        ax.set_ylabel("Phase (degrees)")
        ax.set_ylim(-180, 180)
        ax.set_yticks(np.arange(-180, 181, 90))
    else:
        ax.set_ylabel("Phase (radians)")
        ax.set_ylim(-np.pi, np.pi)
    markerline, stemline, baseline = ax.stem(
        freq_bins,
        phase_spectrum,
        linefmt="g-",
        markerfmt="o",
        basefmt=" ",
    )
    markerline.set_markerfacecolor("none")


def plot_reconstruct_signal(ax, t, signal, spectrum):
    reconstructed_signal = np.fft.ifft(spectrum)
    expanded_reconstruct_signal = np.tile(
        reconstructed_signal, duration // len(reconstructed_signal) + 1
    )[:duration]
    ax.plot(t, signal, label="Original Signal")
    ax.stem(
        np.arange(len(expanded_reconstruct_signal)),
        expanded_reconstruct_signal.real,
        label="Reconstructed Signal",
    )
    ax.set_title("Signal Reconstruction")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Amplitude")
    ax.grid()


_ = interact(
    interactive_update,
    sampling_duration_equal_to_period=slider,
)

interactive(children=(IntSlider(value=10, description='采样点数', layout=Layout(width='500px'), max=50, min=5), Ou…